In [2]:
import pandas as pd

def top_entailment_per_target(df):
    """
    For each target, select the row with the highest 'entailment' score.
    
    Parameters:
        df (pd.DataFrame): DataFrame with columns including 'target' and 'entailment'.
        
    Returns:
        pd.DataFrame: DataFrame with one row per target, having the highest entailment score.
    """
    # For each target, find index of row with max entailment
    idx = df.groupby('target')['entailment'].idxmax(axis=0)


    return df.loc[idx].reset_index(drop=True)

# Example usage
data = {
    'textid': ['greeting', 'question', 'greeting', 'question','greeting', 'question'],
    'target': [0, 0, 1, 1,2,2],
    'model': ['huggingface/distilbert-base-uncased-finetuned-mnli']*6,
    'tokenizer': ['huggingface/distilbert-base-uncased-finetuned-mnli']*6,
    'predicted': ['entailment', 'entailment', 'contradiction', 'contradiction', 'contradiction', 'contradiction'],
    'prob': [0.8553178310394287, 0.8553178310394287, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032],
    'entailment': [0.8553178310394287, 0.8553178310394287, 0.31023797392845154, 0.31023797392845154, 0.31023797392845154, 0.31023797392845154],
    'neutral': [0.13772304356098175, 0.13772304356098175, 0.3048897087574005, 0.3048897087574005, 0.3048897087574005, 0.3048897087574005],
    'contradiction': [0.006959038320928812, 0.006959038320928812, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032]
}

df = pd.DataFrame(data)
top_df = top_entailment_per_target(df)
display(top_df)



/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/93677516.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')['entailment'].idxmax(axis=0)


,textid,target,model,tokenizer,predicted,prob,entailment,neutral,contradiction
0,greeting,0,huggingface/distilbert-base-uncased-finetuned-...,huggingface/distilbert-base-uncased-finetuned-...,entailment,0.855318,0.855318,0.137723,0.006959
1,greeting,1,huggingface/distilbert-base-uncased-finetuned-...,huggingface/distilbert-base-uncased-finetuned-...,contradiction,0.384872,0.310238,0.304890,0.384872
2,greeting,2,huggingface/distilbert-base-uncased-finetuned-...,huggingface/distilbert-base-uncased-finetuned-...,contradiction,0.384872,0.310238,0.304890,0.384872


In [3]:
import pandas as pd
import numpy as np

def get_positive_example(df, percent=0.1, score_cols=['entailment', 'neutral', 'contradiction']):
    """
    From the top entailment per target, select the top X% most confident examples 
    based on the delta between the top and second-highest MNLI scores.
    
    Parameters:
        df (pd.DataFrame): DataFrame with MNLI score columns.
        percent (float): Fraction of top examples to return (0 < percent <= 1)
        score_cols (list): Names of MNLI score columns to consider
        
    Returns:
        pd.DataFrame: DataFrame with top X% most confident examples.
    """
    # First, select top entailment per target
    top_df = top_entailment_per_target(df)
    
    # Extract MNLI score values
    scores = top_df[score_cols]
    print(  scores)
    #Maximum values per row:
    largeset_score = np.max(scores,axis = 1)
    # make it so that 
    # all value before -2 are less thanit and all value after it are greater, so we specify that there is only index -1 greater than it
    second_largest = pd.Series([np.partition(scores.iloc[i], -2)[-2] for i in range(len(scores))])
    print("largest\n")
    print(largeset_score)
    print("second largest\n")
    print( second_largest )
    
    delta =  largeset_score-second_largest 
    print("delta\n")
    print( delta )
    
    # this gets us the value on the top
    
    top_df["delta"]= delta
    
    # the smaller index the greater
    df_sorted_delta = top_df.sort_values(by='delta', ascending=False)
    
    top_percent_df = df_sorted_delta[:int(len(df_sorted_delta)*percent)]

    return top_percent_df
    

    


# Only MNLI scores matter
mnli_labels = ['entailment', 'neutral', 'contradiction']



# Get top 10% most confident rows

print("positive\n")
display(get_positive_example(df, percent=1, score_cols=mnli_labels))

positive

   entailment   neutral  contradiction
0    0.855318  0.137723       0.006959
1    0.310238  0.304890       0.384872
2    0.310238  0.304890       0.384872
largest

0    0.855318
1    0.384872
2    0.384872
dtype: float64
second largest

0    0.137723
1    0.310238
2    0.310238
dtype: float64
delta

0    0.717595
1    0.074634
2    0.074634
dtype: float64


/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/93677516.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')['entailment'].idxmax(axis=0)


,textid,target,model,tokenizer,predicted,prob,entailment,neutral,contradiction,delta
0,greeting,0,huggingface/distilbert-base-uncased-finetuned-...,huggingface/distilbert-base-uncased-finetuned-...,entailment,0.855318,0.855318,0.137723,0.006959,0.717595
1,greeting,1,huggingface/distilbert-base-uncased-finetuned-...,huggingface/distilbert-base-uncased-finetuned-...,contradiction,0.384872,0.310238,0.304890,0.384872,0.074634
2,greeting,2,huggingface/distilbert-base-uncased-finetuned-...,huggingface/distilbert-base-uncased-finetuned-...,contradiction,0.384872,0.310238,0.304890,0.384872,0.074634


In [ ]:
# this is just redefining the same function without the print so it is less annoying 

def get_positive_example(df, percent=0.1, score_cols=['entailment', 'neutral', 'contradiction']):
    """
    From the top entailment per target, select the top X% most confident examples base on delta
    based on the delta between the top and second-highest MNLI scores.
    
    Parameters:
        df (pd.DataFrame): DataFrame with MNLI score columns.
        percent (float): Fraction of top examples to return (0 < percent <= 1)
        score_cols (list): Names of MNLI score columns to consider
        
    Returns:
        pd.DataFrame: DataFrame with top X% most confident examples.
    """
    # First, select top entailment per target
    top_df = top_entailment_per_target(df)
    
    # Extract MNLI score values
    scores = top_df[score_cols]
    #Maximum values per row:
    largeset_score = np.max(scores,axis = 1)
    # make it so that 
    # all value before -2 are less thanit and all value after it are greater, so we specify that there is only index -1 greater than it
    second_largest = pd.Series([np.partition(scores.iloc[i], -2)[-2] for i in range(len(scores))])
  
    
    delta =  largeset_score-second_largest 
  
  
    # this gets us the value on the top
    
    top_df["delta"]= delta
    
    # the smaller index the greater
    df_sorted_delta = top_df.sort_values(by='delta', ascending=False)
    
    top_percent_df = df_sorted_delta[:int(len(df_sorted_delta)*percent)]

    return top_percent_df
    

In [ ]:
    
def get_negative_random(positve,contra = "CONTRADICTION"):
    """
    For each entailment pair, generate a negative example by replacing the class
    in the hypothesis with a random different class, and assign the contradict label.
    
    Parameters:
    
       Positve df
        
    Returns:
       Full finetuning dataset
    """
    new_data_frame = []
    for idx in range(len(positive)):
        pos = positive.iloc[idx]
        pos["predicted"] = contra 
        pos["predicted"] = pd.sample([])
        
        mapping_mask =  {0:"World" ,1:"Sports",2:"Business",3:"Sci/Tech"}

        text_classfication_true["predicted"] =  Max_entailment["textid"].map(mapping_mask)
  
        
        
        
    # pd.concat([])
    

positive = get_positive_example(df, percent=0.5, score_cols=mnli_labels)

get_negative_random(positive)
    


textid                                                    greeting
target                                                           0
model            huggingface/distilbert-base-uncased-finetuned-...
tokenizer        huggingface/distilbert-base-uncased-finetuned-...
predicted                                               entailment
prob                                                      0.855318
entailment                                                0.855318
neutral                                                   0.137723
contradiction                                             0.006959
delta                                                     0.717595
Name: 0, dtype: object


/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/93677516.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')['entailment'].idxmax(axis=0)
